# VisionGym Baseline
Colab GPU에서 synthetic benchmark를 생성하고 Qwen3-VL-2B-Instruct를 zero-shot / prompt variant로 평가합니다.

In [ ]:
# 발표 및 재현성을 위해 실제 할당 GPU 정보를 기록합니다.
import subprocess
subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv,noheader'], check=True)

In [ ]:
# 공개 저장소를 Colab에 가져오고 VLM inference 의존성을 설치합니다.
!git clone -q https://github.com/oosuhada/visiongym.git /content/visiongym || true
%cd /content/visiongym
!git pull -q
!pip install -q -e '.[vlm]'

In [ ]:
# 데이터 생성은 CPU 작업이므로 GPU 시간을 쓰기 전에 완료합니다.
!visiongym generate --config configs/dataset.yaml --output data/generated
import json
from pathlib import Path
manifest = json.loads(Path('data/generated/manifest.json').read_text())
print('benchmark QA:', manifest['benchmark_qa_pairs'])
for split in manifest['splits']:
    print(split['split'], split['scenes'], split['qa_pairs'])

In [ ]:
# 기본 direct-answer prompt로 전체 ID + OOD benchmark를 평가합니다.
!visiongym infer --dataset data/generated/benchmark.jsonl --output outputs/base-direct.jsonl --model Qwen/Qwen3-VL-2B-Instruct --prompt-mode direct --load-in-4bit --batch-size 8
!visiongym evaluate --dataset data/generated/benchmark.jsonl --predictions outputs/base-direct.jsonl --output reports/base-direct --model Qwen/Qwen3-VL-2B-Instruct --prompt-mode direct
!visiongym report --metrics reports/base-direct/metrics.json --output reports/base-direct

In [ ]:
# JSON output prompt를 독립 셀에서 평가해 실패 시 재시작 범위를 줄입니다.
!visiongym infer --dataset data/generated/benchmark.jsonl --output outputs/base-json.jsonl --model Qwen/Qwen3-VL-2B-Instruct --prompt-mode json --load-in-4bit --batch-size 8
!visiongym evaluate --dataset data/generated/benchmark.jsonl --predictions outputs/base-json.jsonl --output reports/base-json --model Qwen/Qwen3-VL-2B-Instruct --prompt-mode json

In [ ]:
# 내부 추론 지시 prompt를 평가합니다.
!visiongym infer --dataset data/generated/benchmark.jsonl --output outputs/base-reasoning.jsonl --model Qwen/Qwen3-VL-2B-Instruct --prompt-mode reasoning --load-in-4bit --batch-size 8
!visiongym evaluate --dataset data/generated/benchmark.jsonl --predictions outputs/base-reasoning.jsonl --output reports/base-reasoning --model Qwen/Qwen3-VL-2B-Instruct --prompt-mode reasoning

In [ ]:
# Canonical answer 예시를 포함한 few-shot prompt를 평가합니다.
!visiongym infer --dataset data/generated/benchmark.jsonl --output outputs/base-fewshot.jsonl --model Qwen/Qwen3-VL-2B-Instruct --prompt-mode fewshot --load-in-4bit --batch-size 8
!visiongym evaluate --dataset data/generated/benchmark.jsonl --predictions outputs/base-fewshot.jsonl --output reports/base-fewshot --model Qwen/Qwen3-VL-2B-Instruct --prompt-mode fewshot

In [ ]:
# 네 prompt의 실제 측정값을 비교하고 발표용 표를 출력합니다.
!visiongym compare reports/base-direct/metrics.json reports/base-json/metrics.json reports/base-reasoning/metrics.json reports/base-fewshot/metrics.json --output reports/prompt-comparison.csv
import pandas as pd
display(pd.read_csv('reports/prompt-comparison.csv'))

In [ ]:
# 결과, predictions, benchmark를 한 번에 내려받을 수 있게 묶습니다.
from pathlib import Path
import shutil
bundle_root = Path('/content/visiongym-baseline-artifacts')
if bundle_root.exists():
    shutil.rmtree(bundle_root)
for source in [Path('reports'), Path('outputs'), Path('data/generated')]:
    shutil.copytree(source, bundle_root / source, dirs_exist_ok=True)
archive = shutil.make_archive(str(bundle_root), 'zip', root_dir=bundle_root)
print('artifact bundle:', archive)
from google.colab import files
files.download(archive)